In [3]:
# Cell 1 - Install & imports
!pip install lightgbm==4.1.0 scikit-learn pandas numpy joblib tqdm pillow opencv-python --quiet

import os, zipfile, re, time
import numpy as np, pandas as pd
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
from scipy import sparse
from PIL import Image, ImageStat
import cv2
from google.colab import files


In [4]:
# Cell 2 - Upload
print("Upload dataset zip (the amazonml zip you uploaded previously)")
uploaded = files.upload()
zip_path = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('./dataset_root')
print("Extracted. Dataset folder:")
!ls ./dataset_root/student_resource/dataset


Upload dataset zip (the amazonml zip you uploaded previously)


Saving 68e8d1d70b66d_student_resource.zip to 68e8d1d70b66d_student_resource (3).zip
Extracted. Dataset folder:
sample_test.csv  sample_test_out.csv  test.csv	train.csv


In [5]:
# Cell 3 - helpers
import math
def extract_fields_from_catalog(text):
    if pd.isna(text): return {'title':'','description':'','value':np.nan,'unit':'','ipq':np.nan}
    title=''; desc=[]; value=np.nan; unit=''; ipq=np.nan
    for line in text.splitlines():
        line=line.strip()
        if not line: continue
        low=line.lower()
        if low.startswith('item name:'):
            title=line.split(':',1)[1].strip()
        elif low.startswith('value:'):
            nums=re.findall(r"[-+]?[0-9]*\.?[0-9]+", line)
            if nums: value=float(nums[0])
        elif low.startswith('unit:'):
            unit=line.split(':',1)[1].strip()
        else:
            desc.append(line)
    description=' '.join(desc)
    m=re.search(r'pack of (\d+)', text.lower());
    if m: ipq=int(m.group(1))
    return {'title':title,'description':description,'value':value,'unit':unit,'ipq':ipq}

def clean_text(s):
    if pd.isna(s): return ''
    s = s.lower()
    s = re.sub(r'[^a-z0-9 ]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom[denom == 0] = 1
    return np.mean(np.abs(y_true - y_pred) / denom)

def cv_target_encode(train_series, target, n_splits=5, smoothing=1.0, noise_level=1e-6):
    """
    Returns oof_encoded (train) and encoded_apply (function to encode test)
    smoothing controls blend between global mean and category mean.
    """
    df = pd.DataFrame({ 'cat': train_series.values, 'target': target })
    global_mean = target.mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof = np.zeros(len(df))
    counts = {}
    sums = {}
    for tr_idx, val_idx in kf.split(df):
        agg = df.iloc[tr_idx].groupby('cat')['target'].agg(['sum','count'])
        sums_fold = agg['sum'].to_dict()
        counts_fold = agg['count'].to_dict()
        for i in val_idx:
            cat = df.iloc[i]['cat']
            s = sums_fold.get(cat, 0.0)
            c = counts_fold.get(cat, 0)
            enc = (s + smoothing*global_mean) / (c + smoothing)
            oof[i] = enc + noise_level * np.random.randn()
    # For test: compute full-train encoding
    agg_full = df.groupby('cat')['target'].agg(['sum','count'])
    sums = agg_full['sum'].to_dict()
    counts = agg_full['count'].to_dict()
    def encode_series(series):
        res = []
        for cat in series.values:
            s = sums.get(cat, 0.0)
            c = counts.get(cat, 0)
            enc = (s + smoothing*global_mean) / (c + smoothing)
            res.append(enc)
        return np.array(res)
    return oof, encode_series


In [6]:
# Cell 4 - load & parse
data_dir = './dataset_root/student_resource/dataset'
train = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test  = pd.read_csv(os.path.join(data_dir, 'test.csv'))
print("train/test rows:", len(train), len(test))

for df in (train, test):
    parsed = df['catalog_content'].fillna('').apply(extract_fields_from_catalog)
    parsed_df = pd.DataFrame(parsed.tolist(), index=df.index)
    for c in parsed_df.columns:
        df[c] = parsed_df[c]

def add_basic(df):
    df['title'] = df['title'].map(clean_text)
    df['description'] = df['description'].map(clean_text)
    df['title_len'] = df['title'].str.len()
    df['desc_len'] = df['description'].str.len()
    df['has_image'] = df['image_link'].notna().astype(int)
    df['value_filled'] = df['value'].fillna(-1.0)
    df['ipq_filled'] = df['ipq'].fillna(1.0)
    df['unit_short'] = df['unit'].str.lower().str.replace('[^a-z0-9 ]','',regex=True).str.split().str[0].fillna('')
    df['brand_guess'] = df['title'].str.split().str[0].fillna('')
    return df

train = add_basic(train)
test  = add_basic(test)


train/test rows: 75000 75000


In [21]:
# Cell 5 - target encoding (train oof and test encode functions)
print("Target-encoding unit and brand (CV-safe)...")
oof_unit, encode_unit_fn = cv_target_encode(train['unit_short'], train['price'], n_splits=5, smoothing=10.0)
oof_brand, encode_brand_fn = cv_target_encode(train['brand_guess'], train['price'], n_splits=5, smoothing=50.0)

train['unit_te'] = oof_unit
train['brand_te'] = oof_brand
test['unit_te']  = encode_unit_fn(test['unit_short'])
test['brand_te'] = encode_brand_fn(test['brand_guess'])
print("Done: sample unit_te, brand_te ->", train[['unit_te','brand_te']].head(3).to_dict(orient='list'))


Target-encoding unit and brand (CV-safe)...
Done: sample unit_te, brand_te -> {'unit_te': [21.039460572955406, 21.589067381953218, 21.452873180428515], 'brand_te': [16.457114901063335, 23.34083735512621, 19.694377291192747]}


In [7]:
# Cell 6 - quick image features (fast). If no images, columns will be NaN -> fill with 0.
from io import BytesIO
import requests, math, numpy as np

def compute_image_features(url):
    try:
        # download with requests (colab supports internet)
        r = requests.get(url, timeout=10)
        img = Image.open(BytesIO(r.content)).convert('RGB')
        arr = np.array(img)
        # mean RGB
        mean_rgb = arr.mean(axis=(0,1))  # [R,G,B]
        # convert to gray and compute Canny edge density
        gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
        edges = cv2.Canny(gray, 100, 200)
        edge_density = edges.mean()
        # size
        h,w = arr.shape[:2]
        return float(mean_rgb[0]), float(mean_rgb[1]), float(mean_rgb[2]), float(edge_density), float(h*w)
    except Exception as e:
        return np.nan, np.nan, np.nan, np.nan, np.nan

# Compute on a sample (do in a loop; this will take time but is usually okay for 75k; you can compute on train only first)
def add_img_feats(df, n_sample=None):
    n = len(df) if n_sample is None else min(n_sample, len(df))
    cols = ['img_r','img_g','img_b','img_edge_density','img_area']
    for c in cols:
        df[c] = np.nan
    for i in range(n):
        url = df.iloc[i]['image_link']
        if not isinstance(url, str) or url.strip()=='':
            continue
        r = compute_image_features(url)
        df.iat[i, df.columns.get_loc('img_r')] = r[0]
        df.iat[i, df.columns.get_loc('img_g')] = r[1]
        df.iat[i, df.columns.get_loc('img_b')] = r[2]
        df.iat[i, df.columns.get_loc('img_edge_density')] = r[3]
        df.iat[i, df.columns.get_loc('img_area')] = r[4]
    # fill NA with zeros (so models don't choke)
    df['img_r'] = df['img_r'].fillna(0.0)
    df['img_g'] = df['img_g'].fillna(0.0)
    df['img_b'] = df['img_b'].fillna(0.0)
    df['img_edge_density'] = df['img_edge_density'].fillna(0.0)
    df['img_area'] = df['img_area'].fillna(0.0)
    return df

# Warning: this can be the slowest step — compute only on train and test fully if you have time.
print("Computing image features (this may take time). If you want to skip, set do_images=False")
do_images = False
if do_images:
    train = add_img_feats(train, n_sample=1000)  # only first 1000 rows
    test  = add_img_feats(test, n_sample=1000)

else:
    for c in ['img_r','img_g','img_b','img_edge_density','img_area']:
        train[c]=0.0; test[c]=0.0
print("Image features done. sample:", train[['img_r','img_edge_density']].head(2).to_dict(orient='list'))


Computing image features (this may take time). If you want to skip, set do_images=False
Image features done. sample: {'img_r': [0.0, 0.0], 'img_edge_density': [0.0, 0.0]}


In [8]:
# Cell 7 - TF-IDF + SVD
train_texts = (train['title'].fillna('') + ' ' + train['description'].fillna('')).values
test_texts  = (test['title'].fillna('') + ' ' + test['description'].fillna('')).values

tf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=3)
t0=time.time()
tf.fit(train_texts)
X_text_train = tf.transform(train_texts)
X_text_test  = tf.transform(test_texts)
print("TFIDF built in", round(time.time()-t0,1), "s; shape:", X_text_train.shape)

svd = TruncatedSVD(n_components=250, random_state=42)
t0=time.time()
X_svd_train = svd.fit_transform(X_text_train)
X_svd_test  = svd.transform(X_text_test)
print("SVD done in", round(time.time()-t0,1), "s; SVD shape:", X_svd_train.shape)


TFIDF built in 68.3 s; shape: (75000, 20000)
SVD done in 59.5 s; SVD shape: (75000, 250)


In [9]:
# Cell 8 - prepare matrices (include image + target-encoded + numeric)
# numeric fields
num_cols = ['title_len','desc_len','has_image','value_filled','ipq_filled','unit_te','brand_te','img_r','img_g','img_b','img_edge_density','img_area']
# make sure existing
for c in num_cols:
    if c not in train.columns:
        train[c]=0.0; test[c]=0.0

scaler = StandardScaler()
scaler.fit(train[num_cols])
X_num_train = scaler.transform(train[num_cols])
X_num_test  = scaler.transform(test[num_cols])

# unit ohe (small)
try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)
ohe.fit(train[['unit_short']])
X_unit_train = ohe.transform(train[['unit_short']])
X_unit_test  = ohe.transform(test[['unit_short']])

# linear dense stack: SVD + num + unit_ohe
X_linear_train = np.hstack([X_svd_train, X_num_train, X_unit_train])
X_linear_test  = np.hstack([X_svd_test,  X_num_test,  X_unit_test])

# LGB uses sparse combined (SVD as dense -> convert to sparse for memory)
X_lgb_train = sparse.hstack([sparse.csr_matrix(X_svd_train), sparse.csr_matrix(X_num_train), sparse.csr_matrix(X_unit_train)]).tocsr()
X_lgb_test  = sparse.hstack([sparse.csr_matrix(X_svd_test),  sparse.csr_matrix(X_num_test),  sparse.csr_matrix(X_unit_test)]).tocsr()

y = train['price'].values
y_log = np.log1p(y)

print("Shapes linear/lgb:", X_linear_train.shape, X_lgb_train.shape)


Shapes linear/lgb: (75000, 317) (75000, 317)


In [10]:
# Cell 9 - Ridge OOF
kf = KFold(n_splits=4, shuffle=True, random_state=42)
ridge_oof = np.zeros(len(train)); ridge_test = np.zeros(len(test))

for fold, (tr, val) in enumerate(kf.split(X_linear_train)):
    print("Ridge fold", fold+1)
    model = Ridge(alpha=10.0)
    model.fit(X_linear_train[tr], np.log1p(y[tr]))
    val_pred = model.predict(X_linear_train[val])
    ridge_oof[val] = np.expm1(val_pred)
    ridge_test += np.expm1(model.predict(X_linear_test)) / kf.n_splits

print("Ridge CV SMAPE:", smape(y, ridge_oof))


Ridge fold 1
Ridge fold 2
Ridge fold 3
Ridge fold 4
Ridge CV SMAPE: 0.6048843789194077


In [11]:
# Cell 10 - LightGBM
kf = KFold(n_splits=4, shuffle=True, random_state=42)
lgb_oof = np.zeros(len(train)); lgb_test = np.zeros(len(test))

lgb_params = {
    'objective':'regression',
    'metric':'mae',
    'learning_rate':0.05,
    'num_leaves':127,
    'min_data_in_leaf':20,
    'feature_fraction':0.7,
    'bagging_fraction':0.7,
    'bagging_freq':5,
    'lambda_l1':0.2,
    'lambda_l2':0.2,
    'seed':42,
    'verbose':-1
}

for fold, (tr, val) in enumerate(kf.split(X_lgb_train)):
    print("LGB fold", fold+1)
    dtrain = lgb.Dataset(X_lgb_train[tr], label=y_log[tr])
    dval = lgb.Dataset(X_lgb_train[val], label=y_log[val])
    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=1200,
        valid_sets=[dtrain, dval],
        callbacks=[lgb.early_stopping(stopping_rounds=80), lgb.log_evaluation(100)]
    )
    val_pred = model.predict(X_lgb_train[val], num_iteration=model.best_iteration)
    lgb_oof[val] = np.expm1(val_pred)
    lgb_test += np.expm1(model.predict(X_lgb_test, num_iteration=model.best_iteration)) / kf.n_splits

print("LGB CV SMAPE:", smape(y, lgb_oof))


LGB fold 1
Training until validation scores don't improve for 80 rounds
[100]	training's l1: 0.46925	valid_1's l1: 0.560694
[200]	training's l1: 0.390784	valid_1's l1: 0.542931
[300]	training's l1: 0.335671	valid_1's l1: 0.536483
[400]	training's l1: 0.291712	valid_1's l1: 0.532289
[500]	training's l1: 0.256051	valid_1's l1: 0.52966
[600]	training's l1: 0.225541	valid_1's l1: 0.528031
[700]	training's l1: 0.199651	valid_1's l1: 0.527014
[800]	training's l1: 0.176699	valid_1's l1: 0.525935
[900]	training's l1: 0.156884	valid_1's l1: 0.525135
[1000]	training's l1: 0.139791	valid_1's l1: 0.524542
[1100]	training's l1: 0.124612	valid_1's l1: 0.524008
[1200]	training's l1: 0.111564	valid_1's l1: 0.523566
Did not meet early stopping. Best iteration is:
[1199]	training's l1: 0.111686	valid_1's l1: 0.523557
LGB fold 2
Training until validation scores don't improve for 80 rounds
[100]	training's l1: 0.472313	valid_1's l1: 0.551254
[200]	training's l1: 0.392312	valid_1's l1: 0.534219
[300]	train

In [12]:
# Cell 11 - stacking meta-model (RidgeCV)
from sklearn.model_selection import cross_val_predict
X_meta_train = np.vstack([ridge_oof, lgb_oof]).T
X_meta_test  = np.vstack([ridge_test, lgb_test]).T

meta = RidgeCV(alphas=[0.1,1.0,10.0], cv=4)
meta.fit(X_meta_train, np.log1p(y))  # fit on log target
meta_oof_pred = np.expm1(meta.predict(X_meta_train))
meta_test_pred = np.expm1(meta.predict(X_meta_test))

print("Meta CV SMAPE:", smape(y, meta_oof_pred))


Meta CV SMAPE: 0.5814048454764605


/tmp/ipython-input-1326593668.py:9: RuntimeWarning: overflow encountered in expm1
  meta_test_pred = np.expm1(meta.predict(X_meta_test))


In [13]:
# Cell 12 - save submission
sub = pd.DataFrame({'sample_id': test['sample_id'], 'price': meta_test_pred})
sub.to_csv('test_out_advanced.csv', index=False)
print("Saved test_out_advanced.csv with shape", sub.shape)
files.download('test_out_advanced.csv')


Saved test_out_advanced.csv with shape (75000, 2)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>